# Provisional error analysis

Summarizes the 30 traceable rule-assigned failures. Native-speaker review is required before paper claims.

**Status:** provisional development evidence; zero locked test queries used.


In [1]:
from pathlib import Path
import csv, hashlib, json, random, statistics
from collections import Counter
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SEED = 20250816
random.seed(SEED)
print(f"Project: {ROOT.name} | fixed seed: {SEED}")


Project: RAABTA_PROJECT_PORTABLE | fixed seed: 20250816


In [2]:
with (ROOT / "reports/error_analysis/provisional_failures_30.csv").open(encoding="utf-8-sig", newline="") as handle:
    failures = list(csv.DictReader(handle))
categories = {}
for row in failures:
    categories[row["category"]] = categories.get(row["category"], 0) + 1
assert len(failures) >= 30
print(json.dumps({"failures": len(failures), "categories": categories, "review_status": "human review pending"}, ensure_ascii=False, indent=2))


{
  "failures": 30,
  "categories": {
    "Roman spelling mismatch": 5,
    "code-switching failure": 5,
    "excessive spelling noise": 5,
    "irrelevant retrieval": 5,
    "named-entity mismatch": 5,
    "short ambiguous query": 5
  },
  "review_status": "human review pending"
}


## Failure-category distribution


In [3]:
categories = Counter(row['category'] for row in failures)
print({'failures': len(failures), 'categories': dict(categories), 'query_types': dict(Counter(row['query_type'] for row in failures))})
assert all(row['gold_passage_id'] and row['top_retrieved_passage_id'] for row in failures)


{'failures': 30, 'categories': {'Roman spelling mismatch': 5, 'code-switching failure': 5, 'excessive spelling noise': 5, 'irrelevant retrieval': 5, 'named-entity mismatch': 5, 'short ambiguous query': 5}, 'query_types': {'informal_spelling': 3, 'urdu_english_code_switching': 5, 'highly_noisy_roman_urdu': 5, 'clean_roman_urdu': 5, 'named_entity': 5, 'short_query': 2, 'abbreviated_roman_urdu': 2, 'slightly_ambiguous': 3}}


## Traceable examples


In [4]:
for category in sorted(categories):
    row = next(item for item in failures if item['category'] == category)
    print({'category': category, 'query_id': row['query_id'], 'query': row['roman_urdu_query'], 'gold': row['gold_passage_id'], 'retrieved': row['top_retrieved_passage_id'], 'hypothesis': row['possible_future_improvement']})
print('Improvement fields are hypotheses; they are not measured fixes.')


{'category': 'Roman spelling mismatch', 'query_id': 'raabta-002', 'query': 'mshil shokd kia hy', 'gold': '1020118-p0000-a10a4718d3a4', 'retrieved': '1088673-p0010-db9a600b4fa2', 'hypothesis': 'Learn spelling variants from reviewed Roman-Urdu pairs.'}
{'category': 'code-switching failure', 'query_id': 'raabta-005', 'query': 'what is lndo', 'gold': '102535-p0000-8f8e2d9d9e58', 'retrieved': '871961-p0001-cb2ad96955b7', 'hypothesis': 'Detect language per token and preserve English entities/terms.'}
{'category': 'excessive spelling noise', 'query_id': 'raabta-011', 'query': 'bhadri kia h', 'gold': '931572-p0000-bf4bdd9acf90', 'retrieved': '1036361-p0009-a80f70416c69', 'hypothesis': 'Add a confidence-aware character-level normalizer.'}
{'category': 'irrelevant retrieval', 'query_id': 'raabta-009', 'query': 'sif almlok (ktab) kya hai', 'gold': '516777-p0000-9642ab95231d', 'retrieved': '142255-p0018-b92f15685d41', 'hypothesis': 'Improve hard-negative training and corpus-aware term weighting.'}

## Interpretation

Outputs above are measurements from frozen local artifacts. Important limitations must remain attached when reused in the report or viva.
